John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

### Import Packages

In [14]:
# Import packages

library(tidyverse)

### Load and Trim Dataset

In [15]:
# minizord.csv already removed non-songs, post-dawn, unmeasureable songs, acoustic outliers
# see "Data Preprocessing/01 preprocessing.ipynb"
data <- read_csv("../data/minizord.csv")

# Only look at North data for now
data <- data %>% filter(neighbourhood == "north") %>% ungroup()

Rows: 7594 Columns: 52
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (4): ID, IST, neighbourhood, Location_Number
dbl  (45): time_rel_sunrise, UST, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_M...
lgl   (1): RepeatST
dttm  (1): DateTime
date  (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Reformat Data: Row for each possible matching opportunity of focal male

In [16]:
# # Get all unique focal male IDs (excluding NAs)
# focal_ids <- unique(na.omit(data$ID))
# print(focal_ids)

# # Initialize an empty list to store the output dataframes
# focal_outputs <- list()

# # Just focusing on one focal male for now (central male LbOW)
# focal_id <- "LbOW"
# # Get rows in which ID == focal_id
# focals_data <- filter(data, ID == focal_id)

# # Get unique set of USTs for the focal (including NA values)
# focal_repertoire <- unique(filter(data, ID == focal_id)$UST)

# # Find all rows that have a UST in focal's repertoire, not including NA values
# focal_ind <- data %>% filter(UST %in% focal_repertoire & !is.na(UST))
# # Group these rows by date, create columns to show focal's next UST
# focal_ind <- focal_ind %>%
#     group_by(as_date(DateTime)) %>% # Group by date so we're not comparing "last song" across days
#     mutate(focal_next_UST = if_else(ID == focal_id, UST, NA)) %>%
#     fill(focal_next_UST, .direction = "up")

# # Create column focal_matches : binary variable (is there a song type match?)
# # If the focal bird is the singer, it cannot match itself, return 0
# # If the focal bird is not the singer, does the song type match the focal's next song? (within day)
# focal_ind <- focal_ind %>%
#     mutate(focal_matches = if_else(ID == focal_id, NA,
#                                     if_else(focal_next_UST == UST, 1, 0)))

In [17]:
# Just focusing on one focal male for now (central male LbOW)
focal_id <- "LbOW"
focal_match_data <- data

In [18]:
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    for (d in unique(na.omit(filter(data, ID==ids[[i]])$date))) {
        print(d)
        print(nrow(filter(data, ID == ids[[i]], date == d)))
    }
}

[1] "LbOW"
[1] 19125
[1] 5
[1] 19126
[1] 8
[1] 19129
[1] 126
[1] 19130
[1] 106
[1] "RbDgY"
[1] 19128
[1] 180
[1] 19129
[1] 191
[1] 19130
[1] 191
[1] "YKY"
[1] 19125
[1] 128
[1] 19126
[1] 131
[1] 19128
[1] 247
[1] 19129
[1] 214
[1] 19130
[1] 199
[1] "TbTbLg"
[1] 19125
[1] 141
[1] 19126
[1] 131
[1] 19128
[1] 127
[1] 19129
[1] 63
[1] 19130
[1] 59
[1] "DgTbO"
[1] 19125
[1] 58
[1] 19126
[1] 147
[1] 19128
[1] 153
[1] 19129
[1] 123
[1] 19130
[1] 163
[1] "RRO"
[1] 19125
[1] 14
[1] 19126
[1] 82
[1] 19128
[1] 46
[1] 19129
[1] 62
[1] 19130
[1] 30


In [19]:
print(unique(na.omit(data$ID)))

[1] "DgTbO"  "LbOW"   "RRO"    "RbDgY"  "TbTbLg" "YKY"   


In [20]:
# Make column for time since Male N's last song
# NA values at top of each day until that male sings
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    time_last_song <- sprintf("time_M%i_last_song", i)
    time_since_last_song <- sprintf("time_since_M%i_last_song", i)
    print(time_last_song)
    focal_match_data <- focal_match_data %>%
    arrange(DateTime) %>%
    mutate("{time_last_song}" := if_else(ID == ids[[i]], time_rel_sunrise, NA)) %>%
    group_by(date) %>%
    fill({time_last_song}, .direction = "down") %>%
    mutate("{time_since_last_song}" := time_rel_sunrise - !!sym({time_last_song})) %>%
    ungroup()
}

[1] "LbOW"
[1] "time_M1_last_song"
[1] "RbDgY"
[1] "time_M2_last_song"
[1] "YKY"
[1] "time_M3_last_song"
[1] "TbTbLg"
[1] "time_M4_last_song"
[1] "DgTbO"
[1] "time_M5_last_song"
[1] "RRO"
[1] "time_M6_last_song"


In [21]:
slice_sample(focal_match_data, n=5)

ID,DateTime,date,time_rel_sunrise,UST,IST,D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,⋯,time_M2_last_song,time_since_M2_last_song,time_M3_last_song,time_since_M3_last_song,time_M4_last_song,time_since_M4_last_song,time_M5_last_song,time_since_M5_last_song,time_M6_last_song,time_since_M6_last_song
<chr>,<dttm>,<date>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TbTbLg,2022-05-14 05:35:00,2022-05-14,-1259.290,8,TbTbLg_4,NA,53.50176,67.67739,62.95599,⋯,NA,NA,-1271.177,11.887506,-1259.290,0.000000,-1262.468,3.178495,-1259.761,0.4711809
RRO,2022-05-14 05:36:08,2022-05-14,-1191.852,97,RRO_5,NA,53.50176,67.67739,62.95599,⋯,NA,NA,-1192.984,1.131975,-1197.629,5.777080,-1203.293,11.441663,-1191.852,0.0000000
LbOW,2022-05-14 05:23:24,2022-05-14,-1955.078,10,LbOW_1,NA,52.93973,62.79695,62.95599,⋯,NA,NA,-1967.057,11.978942,-1982.115,27.037027,-1965.997,10.918744,NA,NA
YKY,2022-05-17 05:24:22,2022-05-17,-1837.293,17,YKY_5,94.0368,46.47336,61.21936,62.95599,⋯,-1840.813,3.519491,-1837.293,0.000000,-1871.306,34.013397,-1854.168,16.874871,-1887.841,50.5476690
LbOW,2022-05-14 05:32:53,2022-05-14,-1386.524,5,LbOW_2,NA,52.93973,67.67739,62.95599,⋯,NA,NA,-1388.590,2.066795,-1389.097,2.573404,-1393.158,6.634255,-1400.751,14.2269327


In [22]:
# Select only rows with focal male's songs
focal_match_data <- focal_match_data %>%
    filter(ID == focal_id) %>%
# Input variables: M2-M6's last UST, time since M1-M6 UST, # M2-M6's distance
    select(
        "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6",
        UST,
        # TODO: Add M1 last UST, but shift it 1 back (not M1's current song, but M1's previous song)
        "M2_last_UST", "M3_last_UST", "M4_last_UST", "M5_last_UST", "M6_last_UST",
        # time_M1_last_song, time_M2_last_song, time_M3_last_song, time_M4_last_song, time_M5_last_song, time_M6_last_song,
    	time_since_M2_last_song, time_since_M3_last_song, time_since_M4_last_song, time_since_M5_last_song, time_since_M6_last_song, # did not include time since M1 last song because it is always 0
        number_of_matches, does_it_match,
        DateTime, time_rel_sunrise
        # Also consider adding "streak"
        # Also consider adding MX nth last song
    )
# TODO: Make sure UST is a "levels" variable (categorical)
# TODO: If UST not in M1's repertoire, set to NA
focal_repertoire <- unique(filter(data, ID == focal_id)$UST)

In [23]:
# # Select only necessary variables
# focal_ind_lean <- focal_ind %>%
#     select(index, Focal_ID, date, ID, time_rel_sunrise, UST, number_of_matches,
#         does_it_match, focal_matches, dist_to_focal, time_since_focals_last_song,
#         time_focals_last_rend_ST, time_since_focals_last_rend_ST, focal_next_UST:time_since_focals_last_rend_ST) %>%
# # Remove focal's songs (which he cannot match)
#     filter(ID != focal_id) %>%
#     ungroup()

In [24]:
tail(focal_match_data)

D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,UST,M2_last_UST,M3_last_UST,M4_last_UST,M5_last_UST,M6_last_UST,time_since_M2_last_song,time_since_M3_last_song,time_since_M4_last_song,time_since_M5_last_song,time_since_M6_last_song,number_of_matches,does_it_match,DateTime,time_rel_sunrise
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dttm>,<dbl>
78.37111,39.73681,67.67739,62.95599,53.6376,10,27,19,8,10,10,15.556629,8.511716,539.0946,1.5138480,564.5145,2,1,2022-05-18 05:44:35,-624.0167
78.37111,39.73681,67.67739,62.95599,53.6376,9,27,27,8,10,10,1.427092,7.015210,546.2759,8.6951428,571.6958,0,0,2022-05-18 05:44:43,-616.8355
78.37111,39.73681,67.67739,62.95599,53.6376,8,27,17,8,33,10,8.260538,6.385084,553.1094,2.2134972,578.5292,1,1,2022-05-18 05:44:49,-610.0020
78.37111,39.73681,67.67739,62.95599,53.6376,9,19,17,8,10,10,8.279709,8.462771,561.4944,1.4237620,586.9143,0,0,2022-05-18 05:44:58,-601.6170
78.37111,39.73681,67.67739,62.95599,53.6376,8,8,9,8,153,10,6.413591,5.949033,568.9183,0.5918999,594.3381,2,1,2022-05-18 05:45:05,-594.1931
78.37111,39.73681,67.67739,62.95599,53.6376,9,3,8,8,153,10,6.175548,8.935323,579.3218,-0.3307658,604.7416,0,0,2022-05-18 05:45:16,-583.7896


### Export Data

In [25]:
write_csv(focal_match_data, "../data/matching_opportunities.csv")

In [26]:
focal_match_data %>% distinct(UST)

UST
<dbl>
10
5
8
13
120
29
3
9
17
